# Module 2 - Medallion, Gold layer, Kimball and KPI

![Medallion to Power BI](../assets/images/medallion_to_powerbi.png)

The Gold layer is the contract between data engineering, analytics and BI.
This module builds the RetailHub Gold model and makes KPI definitions explicit.

In [ ]:
%run ../setup/00_setup

## Runtime pre-check

Run `data/generate_training_dataset.ipynb` before this notebook.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.dim_date",
]

missing = [table for table in required_tables if not spark.catalog.tableExists(table)]
if missing:
    raise Exception("Missing Gold starter tables. Run data/generate_training_dataset first: " + ", ".join(missing))

print("[OK] Gold starter model exists")

## Why Gold exists

The business value of Gold is not "another table". It is:

- one agreed definition of revenue and margin,
- stable grain for Power BI,
- fewer expensive joins in reports,
- repeatable validation,
- a place to document ownership and refresh.

## Business value case

![Gold business value](../assets/images/gold_business_value.png)

RetailHub case:

The CEO sees lower margin in the weekly dashboard. Without Gold, three teams
arrive with three different answers because they apply different filters and
return logic. With Gold, the group debates the business event, not the SQL.

Training prompt:

- Which KPI must be owned by Finance?
- Which KPI can be owned by Sales Ops?
- Which caveat must be visible in the BI contract?

## Star schema or flat BI table?

![Star schema vs flat table](../assets/images/star_schema_vs_flat_table.png)

For this course we use both ideas:

- dimensions and facts to explain Kimball discipline,
- a curated dashboard table/aggregate for Power BI simplicity and cost control.

## Kimball Gold model

![Kimball Gold model](../assets/images/kimball_gold_model.png)

## Inspect the current Gold grain

The fact table should be one row per order line. Before we create BI objects,
we prove that grain and check whether joins can create fan-out.

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_line_ids,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_line_ids
FROM {GOLD}.fact_sales
""").show()

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows_after_customer_join,
  COUNT(DISTINCT f.line_id) AS distinct_line_ids
FROM {GOLD}.fact_sales f
LEFT JOIN {GOLD}.dim_customer c
  ON f.customer_id = c.customer_id
""").show()

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard AS
SELECT
  f.order_date,
  f.order_id,
  f.line_id,
  f.customer_id,
  c.segment,
  c.region AS customer_region,
  f.product_id,
  p.category,
  p.subcategory,
  f.channel,
  f.status,
  f.quantity,
  f.unit_price,
  f.unit_cost,
  f.discount_pct,
  f.line_revenue,
  f.line_cost,
  f.line_margin,
  CASE WHEN f.status = 'Completed' THEN true ELSE false END AS is_completed,
  CASE WHEN f.status = 'Returned' THEN true ELSE false END AS is_returned
FROM {GOLD}.fact_sales f
LEFT JOIN {GOLD}.dim_customer c ON f.customer_id = c.customer_id
LEFT JOIN {GOLD}.dim_product p ON f.product_id = p.product_id
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard_monthly AS
SELECT
  date_format(order_date, 'yyyy-MM') AS year_month,
  customer_region,
  category,
  channel,
  SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END) AS revenue,
  SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  COUNT(DISTINCT CASE WHEN is_returned THEN order_id END) AS returned_orders
FROM {GOLD}.fact_sales_dashboard
GROUP BY date_format(order_date, 'yyyy-MM'), customer_region, category, channel
""")

print("[OK] Dashboard fact and aggregate created")

## Gold quality gate

![Gold quality gate](../assets/images/gold_quality_gate.png)

The checks below are intentionally business-facing. A red result does not
automatically mean "stop the project"; it means "write a decision".

## KPI dictionary starter

The template lives in `docs/templates/kpi-dictionary.md`.

## KPI definition flow

![KPI definition flow](../assets/images/kpi_definition_flow.png)

Suggested KPI dictionary rows:

| KPI | Business definition | SQL definition | Caveat |
|---|---|---|---|
| Revenue | completed line revenue after discount | `SUM(line_revenue) WHERE is_completed` | excludes returned/cancelled |
| Gross margin | completed revenue minus cost | `SUM(line_margin) WHERE is_completed` | depends on product cost quality |
| Return rate | returned orders / eligible orders | returned / (completed + returned) | distinct count grain matters |

In [ ]:
spark.sql(f"""
SELECT
  SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END) AS revenue,
  SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  COUNT(DISTINCT CASE WHEN is_returned THEN order_id END) AS returned_orders
FROM {GOLD}.fact_sales_dashboard
""").show()

## Distinct-count trap

If the same order can contain products from many categories, summing order
counts by category can over-count orders. This is a good discussion point for
Power BI modelling and aggregation design.

In [ ]:
spark.sql(f"""
WITH by_category AS (
  SELECT
    category,
    COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders
  FROM {GOLD}.fact_sales_dashboard
  GROUP BY category
),
overall AS (
  SELECT COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders
  FROM {GOLD}.fact_sales_dashboard
)
SELECT
  (SELECT SUM(completed_orders) FROM by_category) AS sum_of_category_orders,
  (SELECT completed_orders FROM overall) AS true_completed_orders,
  (SELECT SUM(completed_orders) FROM by_category) - (SELECT completed_orders FROM overall) AS overcount
""").show()

## Reconciliation: detail vs aggregate

The monthly aggregate is safe for summary reporting only if totals reconcile
against the detail table at the same filter scope.

In [ ]:
spark.sql(f"""
WITH detail AS (
  SELECT
    ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
    ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin
  FROM {GOLD}.fact_sales_dashboard
),
agg AS (
  SELECT
    ROUND(SUM(revenue), 2) AS revenue,
    ROUND(SUM(gross_margin), 2) AS gross_margin
  FROM {GOLD}.fact_sales_dashboard_monthly
)
SELECT
  detail.revenue AS detail_revenue,
  agg.revenue AS aggregate_revenue,
  detail.revenue - agg.revenue AS revenue_diff,
  detail.gross_margin AS detail_margin,
  agg.gross_margin AS aggregate_margin,
  detail.gross_margin - agg.gross_margin AS margin_diff
FROM detail CROSS JOIN agg
""").show()

## Data quality score

In [ ]:
quality = spark.sql(f"""
WITH checks AS (
  SELECT 'missing_price' AS check_name, COUNT(*) AS issue_count FROM {GOLD}.fact_sales_dashboard WHERE unit_price IS NULL
  UNION ALL
  SELECT 'invalid_status', COUNT(*) FROM {GOLD}.fact_sales_dashboard WHERE status NOT IN ('Completed','Cancelled','Returned')
  UNION ALL
  SELECT 'future_order_date', COUNT(*) FROM {GOLD}.fact_sales_dashboard WHERE order_date > current_date()
  UNION ALL
  SELECT 'missing_customer', COUNT(*) FROM {GOLD}.fact_sales_dashboard WHERE segment IS NULL
  UNION ALL
  SELECT 'missing_product', COUNT(*) FROM {GOLD}.fact_sales_dashboard WHERE category IS NULL
)
SELECT
  check_name,
  issue_count,
  CASE WHEN issue_count = 0 THEN 20 ELSE greatest(0, 20 - issue_count / 20) END AS score_component
FROM checks
""")

quality.createOrReplaceTempView("dq_components")
quality.show()
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.data_quality_score AS
SELECT
  current_timestamp() AS calculated_at,
  ROUND(SUM(score_component), 1) AS score
FROM dq_components
""")
spark.sql(f"SELECT * FROM {GOLD}.data_quality_score").show()

In [ ]:
spark.sql(f"""
SELECT
  status,
  category,
  order_date,
  unit_price,
  line_revenue,
  line_margin
FROM {GOLD}.fact_sales_dashboard
WHERE unit_price IS NULL
   OR status NOT IN ('Completed','Cancelled','Returned')
   OR order_date > current_date()
ORDER BY order_date DESC
LIMIT 25
""").show(truncate=False)

## Lineage and discoverability

![Catalog Explorer lineage](../assets/images/source_catalog_explorer_lineage.png)

Trainer prompt:

- Which object would you certify?
- Which object would you hide from BI users?
- Which object should have the clearest owner and description?

## Decision card: view vs table vs aggregate

| Option | Use when | Risk |
|---|---|---|
| View | logic is simple and source is small | repeated cost on every query |
| Table | stable BI source, scheduled refresh | needs orchestration |
| Aggregate | summary page or DirectQuery | lower detail, needs grain discipline |

## Optional extension for longer delivery

If the group is strong or the course has 8-9 hours:

- add a Type 2 dimension example for customer segment changes,
- add table and column comments for BI discoverability,
- compare `fact_sales_dashboard` as a view vs table,
- create an additional aggregate by `year_month`, `segment` and `category`,
- ask participants to write the final KPI dictionary row by row.